# TWMD Distillation Experiment (Kaggle T4x2)

This notebook reproduces the `Qwen3-Embedding-4B -> BERT-base` distillation using the newly implemented **Text-Walk Manifold Distillation (TWMD)** method.

**Hardware setup**: Kaggle environment with 2x T4 GPUs. The codebase automatically places the Student model on `cuda:0` and the Teacher model on `cuda:1` to prevent Out-Of-Memory (OOM) errors.

In [ ]:
!pip install -q transformers datasets pandas scipy scikit-learn tqdm

In [ ]:
import os
import torch
import json
from types import SimpleNamespace

# Ensure we are in the AAAI-TALAS repository root. 
# If running on Kaggle and you cloned the repo, you might need to cd into it:
# %cd /kaggle/working/AAAI-TALAS

from distiller import KnowledgeDistiller
from src.evaluation.evaluation_automodel import (
    eval_classification_task, 
    eval_pair_task, 
    eval_sts_task, 
    test_cls_tasks, 
    test_sts_tasks, 
    test_pair_tasks
)

## 1. Configuration & Parameters
Here we define the configuration matching the `Qwen3-Embedding-4B` to `BERT-base` distillation pipeline.

In [ ]:
config_dict = {
    "seed": 42,
    "distill_method": "twmd",
    "student_model_name": "bert-base-uncased",
    "teacher_model_name": "Qwen/Qwen3-Embedding-4B",
    "teacher_dtype": "bfloat16",
    "train_data_path": "data/multi-data/train.csv", # Adjust path if necessary
    "task_type": "pair_cls",
    "max_length": 128,
    
    "batch_size": 32,
    "epochs": 5,
    "learning_rate": 2e-5,
    "min_lr": 1e-6,
    "warmup_ratio": 0.1,
    "save_dir": "./twmd_checkpoints",
    "save_every": 1,
    "save_best": True,
    "num_workers": 2,
    
    # --- Graph & Candidate Parameters ---
    "cache_path": "./cache/teacher_embeddings.pt",
    "heatgeo_cache_path": "./cache/twmd_artifact.pt",
    "pooling_method": "last", # Qwen typical pooling
    "normalize_cache": True,
    "cache_dtype": "float16",
    
    "graph_k": 50,
    "graph_temp": 0.1,
    "diffusion_scales": [1, 2, 4],
    "diffusion_topk": 32,
    "hard_neg_k": 16,
    "random_neg_k": 16,
    "candidate_size": 64,
    "spectral_dim": 16,
    "use_spectral": True,
    
    # --- TWMD Specific Parameters ---
    "num_walks": 1,
    "walk_length": 3,
    "tau_rw": 0.05,
    "lambda_rw_path": 1.0,
    "lambda_diff": 1.0,
    "lambda_spec": 0.01,
    "lambda_anchor": 0.01,
    "student_temp": 0.07,
    "scale_weights": [1.0, 0.1, 0.02]
}

config = SimpleNamespace(**config_dict)

## 2. Train the Model
This will build the Random Walk graphs, perform the distillation (Barycenter Matching + Path-Aware Contrastive Learning), and save the model checkpoints.

In [ ]:
# Initialize distiller
# The codebase handles distributing models to cuda:0 and cuda:1 automatically.
distiller = KnowledgeDistiller(config)

# Start training loop
distiller.train()

## 3. Evaluation on Benchmarks
Evaluate the `bert-base-uncased` student model on Classification, Pair Classification, and STS benchmarks to compare against the results table.

In [ ]:
print("========== EVALUATING TWMD STUDENT ==========")
model = distiller.model_student

print("\n--- 1. Classification Tasks (Banking77, TweetEval, Emotion) ---")
eval_classification_task(model, test_cls_tasks)

print("\n--- 2. Pair Classification Tasks (MRPC, SciTail, WiC) ---")
eval_pair_task(model, test_pair_tasks)

print("\n--- 3. STS Tasks (SICK, STS12, STSB) ---")
eval_sts_task(model, test_sts_tasks)